# 07 · Rewrite addresses with a language model (GPU)

**What this does:** the generator in notebook 06 fills templates, so its text is correct but repetitive. Here an open LLM (large language model) rewrites a share of the addresses in different voices. Every rewrite is fact-checked by code; any rewrite that loses or changes a fact is thrown away and the template text is kept.

**Why a share and not all:** results can then be compared between template text and rewritten text (column `paraphrased`). If a model is much worse on rewritten text, it had learned the templates, not addresses.

**The fact check (`bench/paraphrase.py`) rejects a rewrite if:**
- any number changed (door number, pincode, distance)
- the landmark, street, complex or locality name is no longer there
- the position word changed meaning (for example "opposite" became "behind")
- for "150 m north of X": distance or direction changed

**Model:** `Qwen/Qwen2.5-7B-Instruct` (Apache-2.0 licence), loaded in 4-bit so it fits a free T4 GPU (16 GB). If loading fails, set `MODEL` to `Qwen/Qwen2.5-1.5B-Instruct` and `FOUR_BIT = False`: lower quality rewrites, more rejections, but it runs anywhere.

**Runtime:** roughly 1 hour per 15,000 addresses on a T4. Progress is saved every 2,000 rows, so a disconnected session resumes where it stopped.

**Needs:** notebook 06. This notebook has not been run on a GPU yet: run the trial cell (section 2) first.

In [ ]:
# Setup (the same cell opens every notebook): fetch the project code, install what is missing, find the data folder.
import os, sys, subprocess
REPO = 'SiddharthaRaja/Navigate'
PIP  = ['h3', 'rapidfuzz', 'transformers>=4.44', 'accelerate', 'bitsandbytes']
on_cloud = os.path.exists('/content') or os.path.exists('/kaggle/working')
if on_cloud:
    CODE = '/content/Navigate' if os.path.exists('/content') else '/kaggle/working/Navigate'
    if not os.path.exists(f'{CODE}/bench'):
        token = os.environ.get('GH_TOKEN', '')
        try:                                   # only needed while the GitHub repo is private
            from google.colab import userdata; token = token or userdata.get('GH_TOKEN')
        except Exception: pass
        try:
            from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret('GH_TOKEN')
        except Exception: pass
        url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
        r = subprocess.run(['git', 'clone', '--depth', '1', url, CODE], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        assert r.returncode == 0, 'Could not fetch the code. If the repo is private, add a GitHub token as the secret GH_TOKEN.'
    else:
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'])
    if PIP: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PIP], check=True)
else:
    CODE = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
sys.path.insert(0, CODE)
import numpy as np, pandas as pd
from bench import env
P = env.setup('hyderabad')
print('running on', P['where'], '| data folder:', P['root'])

import torch
assert torch.cuda.is_available() or P['where'] == 'local', 'No GPU: Runtime > Change runtime type > T4 GPU (Colab) or Settings > Accelerator (Kaggle)'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU trial run)')

In [ ]:
MODEL, FOUR_BIT = 'Qwen/Qwen2.5-7B-Instruct', True
TEST_SHARE  = 0.5          # share of generated test rows to rewrite
TRAIN_ROWS  = 30_000       # how many training rows to rewrite (0 = none)
BATCH       = 16

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
tok = AutoTokenizer.from_pretrained(MODEL); tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.eos_token
kw = dict(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')) if FOUR_BIT else dict(torch_dtype=torch.float16)
llm = AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto', **kw).eval()
print('loaded', MODEL)

## 1. Rewrite function

In [ ]:
from bench import paraphrase as pp

@torch.no_grad()
def rewrite(texts, seed=0):
    rng = np.random.default_rng(seed)
    styles = [pp.STYLES[i] for i in rng.integers(0, len(pp.STYLES), len(texts))]
    prompts = [tok.apply_chat_template(pp.build_prompt(t, s), tokenize=False, add_generation_prompt=True) for t, s in zip(texts, styles)]
    enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=96, do_sample=True, temperature=0.8, top_p=0.9, pad_token_id=tok.pad_token_id)
    return [pp.clean_output(s) for s in tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]

def run(df, tag):
    '''Rewrite df.text_clean row by row group, fact-check, and save progress to the cache folder.'''
    cache = f"{P['cache']}/paraphrase_{tag}.parquet"
    done = pd.read_parquet(cache) if os.path.exists(cache) else pd.DataFrame(columns=['id', 'rewrite', 'ok', 'reason'])
    todo = df[~df.id.isin(done.id)]
    rows = []
    for a in range(0, len(todo), BATCH):
        part = todo.iloc[a:a + BATCH]
        outs = rewrite(part.text_clean.tolist(), seed=a)
        for r, o in zip(part.itertuples(), outs):
            ok, why = pp.keep(r.text_clean, o, r.components, r.relation, getattr(r, 'ref_name', None) if isinstance(getattr(r, 'ref_name', None), str) else None)
            rows.append((r.id, o, ok, why))
        if (a // BATCH) % 125 == 124 or a + BATCH >= len(todo):
            done = pd.concat([done, pd.DataFrame(rows, columns=done.columns)], ignore_index=True); rows = []
            done['ok'] = done['ok'].astype(bool)
            done.to_parquet(cache, index=False)
            print(f"{tag}: {len(done):,}/{len(df):,} done, kept {done.ok.mean():.1%}")
    done['ok'] = done['ok'].astype(bool)
    return done

## 2. Trial on 32 rows: read these before the long run
Look at the kept rewrites (do they read like people?) and at the rejected ones (is the fact check being fair?).

In [ ]:
from bench import io
B = P['bench']
test, train = pd.read_parquet(f'{B}/test.parquet'), pd.read_parquet(f'{B}/train.parquet')
gen_rows = test[(test.source == 'synthetic') & ~test.abstain & (test.text_clean.str.len() > 0)]
trial = gen_rows.sample(32, random_state=0)
outs = rewrite(trial.text_clean.tolist()[:16]) + rewrite(trial.text_clean.tolist()[16:], seed=1)
for r, o in zip(trial.itertuples(), outs):
    ok, why = pp.keep(r.text_clean, o, r.components, r.relation, r.ref_name if isinstance(r.ref_name, str) else None)
    print(f"{'KEEP  ' if ok else 'REJECT'} [{why}]\n   in : {r.text_clean}\n   out: {o}")

## 3. Full run

In [ ]:
pick_test  = gen_rows.sample(frac=TEST_SHARE, random_state=1)
done_test  = run(pick_test, 'test')
tr_rows    = train[~train.abstain & (train.text_clean.str.len() > 0)]
done_train = run(tr_rows.sample(min(TRAIN_ROWS, len(tr_rows)), random_state=1), 'train') if TRAIN_ROWS else None

## 4. Merge and save
Kept rewrites replace `text`; the original stays in `text_template`. String-level noise (typos and so on) that the row had is applied again on top of the rewrite, so noise labels stay true.

In [ ]:
from bench import noise as nz

def merge(df, done, seed):
    df = df.copy(); df['text_template'] = df['text']; df['paraphrased'] = False
    if done is None: return df
    good = done[done.ok].set_index('id')['rewrite']
    rng = np.random.default_rng(seed)
    for i in df.index[df.id.isin(good.index)]:
        s = good[df.at[i, 'id']]
        for op in ('abbrev', 'typo', 'merge', 'junk', 'instructions'):
            if op in str(df.at[i, 'noise_ops']).split(','): s = nz.STRING_OPS[op](s, rng)
        df.at[i, 'text'] = s; df.at[i, 'paraphrased'] = True
    return df

test_p, train_p = merge(test, done_test, 1), merge(train, done_train, 2)
test_p.to_parquet(f'{B}/test_paraphrased.parquet', index=False)
train_p.to_parquet(f'{B}/train_paraphrased.parquet', index=False)
print(f"test: {test_p.paraphrased.sum():,} rows rewritten | train: {train_p.paraphrased.sum():,} rows rewritten")
print('rejection reasons (test):'); print(done_test[~done_test.ok].reason.value_counts().head(8).to_string())
for r in test_p[test_p.paraphrased].sample(min(15, int(test_p.paraphrased.sum())), random_state=3).itertuples(): print('  ', r.text)

## Share the outputs
Packs only the files this notebook created into one zip in `My Drive > Navigate > data > share`. Download that one file to share or back up the results.

In [ ]:
env.pack_outputs(P, '07' + '_outputs', [f"{P['bench']}/test_paraphrased.parquet", f"{P['bench']}/train_paraphrased.parquet"])